In [0]:
import requests
import json
from datetime import datetime

# Fetch data from API
response = requests.get("https://api.joinrise.io/api/v1/jobs/")
data = response.json()

# Create filename with current timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
file_path = f"/Volumes/sdp_pipeline/source/job_volume/job_api_scrapper/jobs_{timestamp}.json"

# Save data to JSON file
with open(file_path, "w") as f:
    json.dump(data, f)

In [0]:
df = spark.read.json("/Volumes/sdp_pipeline/source/job_volume/job_api_scrapper/")
display(df)

In [0]:
df.printSchema

In [0]:
from pyspark.sql.functions import explode, col, when, current_date, to_date

jobs_df = df.select(explode(col("result.jobs")).alias("jobs"))

jobs_flat_df = jobs_df.select("jobs.*") \
    .withColumn("created_date", to_date(col("createdAt"))) \
    .withColumn("updated_date", to_date(col("updatedAt"))) \
    .filter((col("created_date") == current_date()) | (col("updated_date") == current_date())).drop("created_date", "updated_date")

jobs_flat_df.write.mode("append").saveAsTable("sdp_pipeline.source.bronze_layer")

In [0]:
silver_df = spark.read.table("sdp_pipeline.source.bronze_layer")

silver_df.printSchema()

In [0]:
from pyspark.sql.functions import explode, col, when, size, coalesce, sha2, concat_ws, lit
df_enriched_silver = silver_df.select(
    col("_id").alias("job_id"),
    col("map.locationName").alias("job_location"),
    col("map.lat").alias("latitude"),
    col("map.lng").alias("longitude"),
    col("title").alias("job_title"),
    col("slug").alias("job_slug"),
    col("url").alias("job_url"),
    col("owner._id").alias("company_id"),
    col("owner.companyName").alias("company_name"),
    col("owner.location").alias("company_location"),
    col("owner.role").alias("company_role"),
    col("type").alias("job_type"),
    col("owner.photo").alias("company_photo"),
    col("updatedAt"),
    col("createdAt"),
    col("owner.benefits.benefits").alias("owner_benefits"),
    col("owner.badges").alias("owner_badges")
).withColumn("owner_benefits", when(size(col("owner_benefits")) > 0, col("owner_benefits")).otherwise(None)).withColumn("owner_badges", when(size(col("owner_badges")) > 0, col("owner_badges")).otherwise(None)).withColumn("job_location_id",
    sha2(coalesce(concat_ws("_", col("latitude").cast("string"), col("longitude").cast("string")), lit("unknown")), 256))


display(df_enriched_silver)

In [0]:
df_enriched_silver.write.mode("append").saveAsTable("sdp_pipeline.source.silver_layer")

In [0]:
dp_gold_full = spark.sql("select * from sdp_pipeline.source.silver_layer")
display(dp_gold_full)

In [0]:
dim_location = spark.sql("select distinct job_location_id,job_location as location,latitude, longitude  from sdp_pipeline.source.silver_layer")
dim_location.write.mode("append").saveAsTable("sdp_pipeline.target.dim_location")

In [0]:
dim_benefits = dp_gold_full.select("job_id",explode("owner_benefits").alias("benefits"))
dim_benefits.write.mode("overwrite").saveAsTable("sdp_pipeline.target.dim_benefits")

In [0]:
dim_badges = dp_gold_full.select("job_id",explode("owner_badges").alias("badges"))
dim_badges.write.mode("overwrite").saveAsTable("sdp_pipeline.target.dim_badges")

In [0]:
dim_company = spark.sql("select distinct company_id,company_name as name,company_location as location,company_role as role,company_photo as photo from sdp_pipeline.source.silver_layer")

dim_company.write.mode("overwrite").saveAsTable("sdp_pipeline.target.dim_company")

In [0]:
dim_job = dp_gold_full.select(col("job_id"),
    col("job_type").alias("type"),
    col("job_title").alias("title"),
    col("job_slug").alias("slug"),
    col("job_url").alias("url"),
    col("createdAt").cast("timestamp").alias("createdAt"),
    col("updatedAt").cast("timestamp").alias("updatedAt")).distinct()

dim_job.write.mode("overwrite").saveAsTable("sdp_pipeline.target.dim_job")

In [0]:
%sql
DROP TABLE IF EXISTS job.source.job_dim_source

In [0]:
# Create fact table with foreign keys to all dimensions
fact_job_posting = spark.sql("""
    SELECT 
        job_id,                    -- FK to dim_job
        company_id,                -- FK to dim_company
        job_location_id,           -- FK to dim_location
        CAST(createdAt as DATE)               -- Date dimension (when job was posted)
                       -- Date dimension (when job was last updated)
    FROM sdp_pipeline.source.silver_layer
""")

# Write to target schema
fact_job_posting.write.mode("overwrite").saveAsTable("sdp_pipeline.target.fact_job_posting")

print(f"Fact table created with {fact_job_posting.count()} records")
display(fact_job_posting)